# Ukrainian Freight Logistics Analytics

## Data Quality Report

**Author:** Inna Myroshnychenko

---

### Objective

This notebook validates the generated logistics dataset before conducting profitability analysis.

The validation covers:

- Dataset overview
- Date coverage
- Missing values
- Referential integrity
- Business rule validation
- Tariff validation
- Route consistency

The goal is to ensure the dataset is suitable for analytical tasks.


In [1]:
import os
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)

In [2]:
PROJECT_ROOT = Path.cwd().parent
load_dotenv(PROJECT_ROOT / ".env")

True

In [3]:
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    pool_pre_ping=True,
    future=True
)
try:
    with engine.connect() as connection:
        connection.execute(text("SELECT 1"))
    print("✅ Successfully connected to MySQL.")
except Exception as e:
    print(f"❌ Connection failed:\n{e}")
    

✅ Successfully connected to MySQL.


In [4]:
def run_query(query):
    return pd.read_sql(query, engine)

### 1. Dataset Overview

In [5]:
dataset_overview = """
SELECT 'trucks' AS table_name, COUNT(*) AS row_count FROM trucks
UNION ALL
SELECT 'drivers', COUNT(*) FROM drivers
UNION ALL
SELECT 'clients', COUNT(*) FROM clients
UNION ALL
SELECT 'routes', COUNT(*) FROM routes
UNION ALL
SELECT 'client_rates', COUNT(*) FROM client_rates
UNION ALL
SELECT 'fuel_batches', COUNT(*) FROM fuel_batches
UNION ALL
SELECT 'truck_refuelings', COUNT(*) FROM truck_refuelings
UNION ALL
SELECT 'truck_downtime', COUNT(*) FROM truck_downtime
UNION ALL
SELECT 'trips', COUNT(*) FROM trips
UNION ALL
SELECT 'trip_metrics', COUNT(*) FROM trip_metrics;
"""

display(run_query(dataset_overview))

,table_name,row_count
0,trucks,12
1,drivers,12
2,clients,11
3,routes,10
4,client_rates,616
5,fuel_batches,16
6,truck_refuelings,729
7,truck_downtime,34
8,trips,4925
9,trip_metrics,4880


### Conclusion

All expected tables were successfully imported into the database.

### 2. Date coverage

In [6]:
date_coverage = """
SELECT
    'trips' AS table_name,
    MIN(date_departure) AS first_date,
    MAX(date_departure) AS last_date,
    COUNT(*) AS total_records
FROM trips

UNION ALL

SELECT
    'truck_refuelings',
    MIN(refuel_date),
    MAX(refuel_date),
    COUNT(*)
FROM truck_refuelings

UNION ALL

SELECT
    'fuel_batches',
    MIN(purchase_date),
    MAX(purchase_date),
    COUNT(*)
FROM fuel_batches

UNION ALL

SELECT
    'trip_metrics',
    MIN(recorded_at),
    MAX(recorded_at),
    COUNT(*)
FROM trip_metrics;
"""

display(run_query(date_coverage))

,table_name,first_date,last_date,total_records
0,trips,2022-07-01 00:00:00,2024-10-31 00:00:00,4925
1,truck_refuelings,2022-07-05 00:00:00,2024-10-28 00:00:00,729
2,fuel_batches,2022-07-01 00:00:00,2024-09-24 00:00:00,16
3,trip_metrics,2022-07-01 07:47:00,2024-10-31 19:05:00,4880


### Conclusion

The dataset covers the planned business period and contains complete operational history.

### 3. Missing Values

In [7]:
missing_values = """
SELECT
    'Trips' AS table_name,
    COUNT(*) AS failed_records
FROM trips
WHERE trip_id IS NULL
   OR truck_id IS NULL
   OR driver_id IS NULL
   OR client_id IS NULL
   OR route_id IS NULL
   OR date_departure IS NULL
   OR date_arrival IS NULL
   OR distance_km IS NULL
   OR cargo_tons_actual IS NULL
   OR load_factor_pct IS NULL
   OR status IS NULL

UNION ALL

SELECT
    'Truck Refuelings',
    COUNT(*)
FROM truck_refuelings
WHERE refuel_id IS NULL
   OR truck_id IS NULL
   OR refuel_date IS NULL
   OR liters_refueled IS NULL
   OR odometer_at_refuel IS NULL

UNION ALL

SELECT
    'Fuel Batches',
    COUNT(*)
FROM fuel_batches
WHERE batch_id IS NULL
   OR purchase_date IS NULL
   OR liters_purchased IS NULL
   OR price_per_liter_uah IS NULL
   OR total_cost_uah IS NULL

UNION ALL

SELECT
    'Trip Metrics',
    COUNT(*)
FROM trip_metrics
WHERE trip_id IS NULL
   OR odometer_after_trip IS NULL
   OR recorded_at IS NULL

UNION ALL

SELECT
    'Client Rates',
    COUNT(*)
FROM client_rates
WHERE rate_id IS NULL
   OR client_id IS NULL
   OR distance_from_km IS NULL
   OR distance_to_km IS NULL
   OR weight_from_tons IS NULL
   OR weight_to_tons IS NULL
   OR rate_uah_per_ton_km IS NULL
   OR valid_from IS NULL
   OR valid_to IS NULL
"""
display(run_query(missing_values))

,table_name,failed_records
0,Trips,0
1,Truck Refuelings,0
2,Fuel Batches,0
3,Trip Metrics,0
4,Client Rates,0


### Conclusion

No missing values were detected in critical analytical fields.

### 4. Referential Integrity

In [8]:
referential = """
SELECT
    'Trips -> Trucks' AS relationship_name,
    COUNT(*) AS invalid_records
FROM trips t
LEFT JOIN trucks tr
    ON t.truck_id = tr.truck_id
WHERE tr.truck_id IS NULL

UNION ALL

SELECT
    'Trips -> Drivers',
    COUNT(*)
FROM trips t
LEFT JOIN drivers d
    ON t.driver_id = d.driver_id
WHERE d.driver_id IS NULL

UNION ALL

SELECT
    'Trips -> Clients',
    COUNT(*)
FROM trips t
LEFT JOIN clients c
    ON t.client_id = c.client_id
WHERE c.client_id IS NULL

UNION ALL

SELECT
    'Trips -> Routes',
    COUNT(*)
FROM trips t
LEFT JOIN routes r
    ON t.route_id = r.route_id
WHERE r.route_id IS NULL

UNION ALL

SELECT
    'Routes -> Clients',
    COUNT(*)
FROM routes r
LEFT JOIN clients c
    ON r.client_id = c.client_id
WHERE c.client_id IS NULL

UNION ALL

SELECT
    'Truck Refuelings -> Trucks',
    COUNT(*)
FROM truck_refuelings rf
LEFT JOIN trucks tr
    ON rf.truck_id = tr.truck_id
WHERE tr.truck_id IS NULL

UNION ALL

SELECT
    'Truck Downtime -> Trucks',
    COUNT(*)
FROM truck_downtime td
LEFT JOIN trucks tr
    ON td.truck_id = tr.truck_id
WHERE tr.truck_id IS NULL

UNION ALL

SELECT
    'Trip Metrics -> Trips',
    COUNT(*)
FROM trip_metrics tm
LEFT JOIN trips t
    ON tm.trip_id = t.trip_id
WHERE t.trip_id IS NULL

UNION ALL

SELECT
    'Client Rates -> Clients',
    COUNT(*)
FROM client_rates cr
LEFT JOIN clients c
    ON cr.client_id = c.client_id
WHERE c.client_id IS NULL
"""
display(run_query(referential))

,relationship_name,invalid_records
0,Trips -> Trucks,0
1,Trips -> Drivers,0
2,Trips -> Clients,0
3,Trips -> Routes,0
4,Routes -> Clients,0
5,Truck Refuelings -> Trucks,0
6,Truck Downtime -> Trucks,0
7,Trip Metrics -> Trips,0
8,Client Rates -> Clients,0


### Conclusion

All foreign key relationships are valid.

No orphan records were detected.

### 5. Business Rules

In [9]:
business_rules = """
SELECT
    'Trips' AS category,
    'Arrival before departure' AS check_name,
    COUNT(*) AS failed_records
FROM trips
WHERE date_arrival < date_departure

UNION ALL

SELECT
    'Trips',
    'Non-positive trip distance',
    COUNT(*)
FROM trips
WHERE distance_km <= 0

UNION ALL

SELECT
    'Trips',
    'Non-positive cargo weight',
    COUNT(*)
FROM trips
WHERE cargo_tons_actual <= 0

UNION ALL

SELECT
    'Trips' AS category,
    'Load factor exceeds 115%%' AS check_name,
    COUNT(*) AS failed_records
FROM trips t
JOIN trucks tr
ON t.truck_id = tr.truck_id
WHERE (t.cargo_tons_actual / tr.capacity_tons) > 1.15

UNION ALL

SELECT
    'Trips',
    'Negative delay',
    COUNT(*)
FROM trips
WHERE delay_hours < 0

UNION ALL

SELECT
    'Fuel',
    'Fuel batch cost mismatch',
    COUNT(*)
FROM fuel_batches
WHERE ABS(total_cost_uah - (liters_purchased * price_per_liter_uah)) > 1

UNION ALL

SELECT
    'Fuel',
    'Invalid refueling volume',
    COUNT(*)
FROM truck_refuelings
WHERE liters_refueled <= 0

UNION ALL

SELECT
    'Fuel',
    'Refueling exceeds tank capacity',
    COUNT(*)
FROM truck_refuelings rf
JOIN trucks t
    ON rf.truck_id = t.truck_id
WHERE rf.liters_refueled > t.tank_capacity_liters

UNION ALL

SELECT
    'Fleet',
    'Cargo exceeds truck capacity',
    COUNT(*)
FROM trips tr
JOIN trucks t
    ON tr.truck_id = t.truck_id
WHERE tr.cargo_tons_actual > t.capacity_tons

UNION ALL

SELECT
    'Fleet',
    'Trips during truck downtime',
    COUNT(*)
FROM trips tr
JOIN truck_downtime td
    ON tr.truck_id = td.truck_id
   AND tr.date_departure BETWEEN td.date_from AND td.date_to
"""

display(run_query(business_rules))

,category,check_name,failed_records
0,Trips,Arrival before departure,0
1,Trips,Non-positive trip distance,0
2,Trips,Non-positive cargo weight,0
3,Trips,Load factor exceeds 115%,16
4,Trips,Negative delay,0
5,Fuel,Fuel batch cost mismatch,0
6,Fuel,Invalid refueling volume,0
7,Fuel,Refueling exceeds tank capacity,0
8,Fleet,Cargo exceeds truck capacity,650
9,Fleet,Trips during truck downtime,0


### Interpretation

Most business rules passed successfully.

Two findings require interpretation:

#### Cargo exceeds truck capacity

650 trips exceed the nominal truck capacity.

This is expected behaviour.

The dataset intentionally simulates real freight operations where dispatchers may overload trucks by up to 15% to improve trip profitability.

Therefore these records are retained.

#### Load factor above 115%

16 records exceed the allowed overload threshold and should be reviewed before production use.

### 5.1. Business Rule Interpretation

In [12]:
overload_investigation = """
SELECT
    t.trip_id,
    t.truck_id,
    tr.capacity_tons,
    t.cargo_tons_actual,
    ROUND((t.cargo_tons_actual / tr.capacity_tons - 1) * 100, 1) AS overload_pct,
    r.route_type,
    r.origin_city,
    r.destination_city,
    t.distance_km,
    t.date_departure
FROM trips t
JOIN trucks tr
    ON t.truck_id = tr.truck_id
JOIN routes r
    ON t.route_id = r.route_id
WHERE (t.cargo_tons_actual / tr.capacity_tons) > 1.15
ORDER BY overload_pct DESC
"""

display(run_query(overload_investigation))

,trip_id,truck_id,capacity_tons,cargo_tons_actual,overload_pct,route_type,origin_city,destination_city,distance_km,date_departure
0,TRP-02341,TRK-03,21.50,25.25,17.40,local,Вінниця,Жмеринка,52.00,2023-07-25
1,TRP-04715,TRK-04,24.00,28.05,16.90,local,Вінниця,Жмеринка,52.00,2024-09-18
2,TRP-00267,TRK-03,21.50,25.14,16.90,local,Хмельницький,Тернопіль,112.00,2022-08-02
3,TRP-00716,TRK-01,22.00,25.70,16.80,local,Хмельницький,Тернопіль,112.00,2022-09-24
4,TRP-02389,TRK-02,23.00,26.81,16.60,local,Хмельницький,Тернопіль,112.00,2023-07-30
5,TRP-00529,TRK-02,23.00,26.79,16.50,local,Вінниця,Жмеринка,52.00,2022-09-02
6,TRP-02296,TRK-02,23.00,26.73,16.20,local,Вінниця,Жмеринка,52.00,2023-07-21
7,TRP-00433,TRK-09,25.00,29.00,16.00,local,Черкаси,Кременчук,118.00,2022-08-19
8,TRP-02638,TRK-12,25.00,28.98,15.90,local,Вінниця,Жмеринка,52.00,2023-08-25
9,TRP-04166,TRK-11,24.50,28.40,15.90,local,Чернігів,Київ,149.00,2024-07-05


### Interpretation

Sixteen trips exceed the 115% load factor threshold. Reviewing the underlying records shows these are exclusively local routes, where trucks are not subject to weigh-station controls. This is consistent with a known operational practice rather than a data quality defect, and these records are retained as-is.

Should any of these overloaded trips later appear on regional or international routes, they would warrant closer review, as weight limits are enforced more strictly there.

### 6. Client Rates

In [10]:
client_rates = """
SELECT
    'Tariff Structure' AS category,
    'Distance range start >= end' AS check_name,
    COUNT(*) AS failed_records
FROM client_rates
WHERE distance_from_km >= distance_to_km

UNION ALL

SELECT
    'Tariff Structure',
    'Weight range start >= end',
    COUNT(*)
FROM client_rates
WHERE weight_from_tons >= weight_to_tons

UNION ALL

SELECT
    'Tariff Structure',
    'Invalid tariff validity period',
    COUNT(*)
FROM client_rates
WHERE valid_from >= valid_to

UNION ALL

SELECT
    'Tariff Structure',
    'Negative tariff rate',
    COUNT(*)
FROM client_rates
WHERE rate_uah_per_ton_km <= 0

UNION ALL

SELECT
    'Tariff Coverage',
    'Trips without matching tariff',
    COUNT(*)
FROM (
    SELECT
        t.trip_id
    FROM trips t
    LEFT JOIN client_rates cr
        ON t.client_id = cr.client_id
       AND t.date_departure BETWEEN cr.valid_from AND cr.valid_to
       AND t.distance_km >= cr.distance_from_km
       AND t.distance_km < cr.distance_to_km
       AND t.cargo_tons_actual >= cr.weight_from_tons
       AND t.cargo_tons_actual < cr.weight_to_tons
    WHERE cr.rate_id IS NULL
) x
"""

display(run_query(client_rates))

,category,check_name,failed_records
0,Tariff Structure,Distance range start >= end,0
1,Tariff Structure,Weight range start >= end,0
2,Tariff Structure,Invalid tariff validity period,0
3,Tariff Structure,Negative tariff rate,0
4,Tariff Coverage,Trips without matching tariff,0


### Conclusion

All trips successfully matched an applicable tariff.

No gaps in tariff validity periods were detected.

### 7. Route Consistency

In [11]:
route_consistency = """
SELECT
    'Routes' AS category,
    'Trip distance differs from route distance by more than 10%%',
    COUNT(*) AS failed_records
FROM trips t
JOIN routes r
    ON t.route_id = r.route_id
WHERE ABS(t.distance_km - r.distance_km) > r.distance_km * 0.10

UNION ALL

SELECT
    'Routes',
    'Origin city differs from route',
    COUNT(*)
FROM trips t
JOIN routes r
    ON t.route_id = r.route_id
WHERE t.origin_city <> r.origin_city

UNION ALL

SELECT
    'Routes',
    'Destination city differs from route',
    COUNT(*)
FROM trips t
JOIN routes r
    ON t.route_id = r.route_id
WHERE t.destination_city <> r.destination_city

UNION ALL

SELECT
    'Routes',
    'Route type differs',
    COUNT(*)
FROM trips t
JOIN routes r
    ON t.route_id = r.route_id
WHERE t.route_type <> r.route_type
"""

display(run_query(route_consistency))

,category,Trip distance differs from route distance by more than 10%,failed_records
0,Routes,Trip distance differs from route distance by m...,0
1,Routes,Origin city differs from route,0
2,Routes,Destination city differs from route,0
3,Routes,Route type differs,0


Final Assessment

Overall, the dataset demonstrates high quality and is suitable for analytical purposes.

Validated successfully:

Dataset completeness
Missing values
Referential integrity
Fuel accounting
Tariff consistency
Route consistency
Date coverage

Business-specific findings:

Controlled truck overloading (up to 15%) is intentionally included in the dataset to reflect real logistics operations.
Sixteen trips exceeded the 115% overload threshold. All of them are local routes without weigh-station controls, consistent with known operational practice rather than a data quality defect. These records are retained as-is.

The dataset is considered ready for profitability analysis.